In [ ]:
# --- 1. SETUP, IMPORTS, AND KAGGLE DOWNLOAD ---
import tensorflow as tf
import os
import time
import matplotlib.pyplot as plt

# Install necessary libraries
!pip install -q datasets transformers sacrebleu kaggle

# Check for GPU
print("TensorFlow version:", tf.__version__)
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

# --- KAGGLE SETUP (Correct Method) ---
# Set up Kaggle API credentials
os.environ['KAGGLE_USERNAME'] = "YOUR_KAGGLE_USERNAME"
os.environ['KAGGLE_KEY'] = "YOUR_KAGGLE_KEY"

DATA_DIR = "/content/datasets"

# Download and unzip the dataset using the official Kaggle command
!kaggle datasets download -d zainuddin123/parallel-corpus-for-english-urdu-language -p {DATA_DIR} --unzip

print("\nDataset downloaded. Contents of the directory are:")
# List the files in the directory to find the correct CSV filename
# This may show a subdirectory, which we will investigate.
!ls {DATA_DIR}

TensorFlow version: 2.19.0
Num GPUs Available:  1
Dataset URL: https://www.kaggle.com/datasets/zainuddin123/parallel-corpus-for-english-urdu-language
License(s): unknown
  0% 0.00/419k [00:00<?, ?B/s]
100% 419k/419k [00:00<00:00, 877MB/s]

Dataset downloaded. Contents of the directory are:
Dataset  data.zip


In [ ]:
!ls {DATA_DIR}/Dataset

english-corpus.txt  urdu-corpus.txt


In [ ]:
# --- 2. DATA PREPROCESSING AND CHECKPOINTING (Final Version for .txt files) ---
from datasets import Dataset, DatasetDict, load_from_disk
from transformers import AutoTokenizer

TOKENIZED_DATASET_PATH = "/content/tokenized_en_ur_dataset"
MODEL_CHECKPOINT = "bert-base-multilingual-cased"

# --- CHECKPOINT LOGIC ---
if os.path.exists(TOKENIZED_DATASET_PATH):
    print(f"Found saved tokenized dataset at '{TOKENIZED_DATASET_PATH}'. Loading from disk...")
    tokenized_datasets = load_from_disk(TOKENIZED_DATASET_PATH)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)
    print("Loading complete.")
else:
    print("No saved dataset found. Starting preprocessing from scratch...")

    # --- Define the paths to our two text files ---
    english_file_path = "/content/datasets/Dataset/english-corpus.txt"
    urdu_file_path = "/content/datasets/Dataset/urdu-corpus.txt"

    # --- Load the parallel text files ---
    with open(english_file_path, "r", encoding="utf-8") as f:
        english_sentences = f.read().splitlines()

    with open(urdu_file_path, "r", encoding="utf-8") as f:
        urdu_sentences = f.read().splitlines()

    # Verify that we have the same number of sentences for each language
    assert len(english_sentences) == len(urdu_sentences), "The number of sentences in both files must be the same."

    # Create a dictionary in the format the 'datasets' library expects
    raw_data = {"English": english_sentences, "Urdu": urdu_sentences}

    # Convert our Python dictionary into a Hugging Face Dataset object
    raw_dataset = Dataset.from_dict(raw_data)

    # Now we can proceed as before
    split = raw_dataset.train_test_split(test_size=0.1, seed=42)
    dataset = DatasetDict({
        'train': split['train'],
        'test': split['test']
    })

    # Load the tokenizer
    tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

    # Define the preprocessing function (this remains the same)
    def preprocess_function(examples):
        inputs = [str(ex) for ex in examples["English"]]
        targets = [str(ex) for ex in examples["Urdu"]]
        model_inputs = tokenizer(inputs, text_target=targets, max_length=128, truncation=True)
        return model_inputs

    print("\nTokenizing the raw dataset...")
    tokenized_datasets = dataset.map(
        preprocess_function, batched=True, remove_columns=dataset["train"].column_names
    )

    print("\nTokenization complete. Saving tokenized dataset to disk...")
    tokenized_datasets.save_to_disk(TOKENIZED_DATASET_PATH)
    print("Dataset saved.")

# --- Verify the final data ---
print("\nFinal tokenized dataset structure:")
print(tokenized_datasets)
print("\nExample of a single tokenized entry:")
print(tokenized_datasets["train"][0])

No saved dataset found. Starting preprocessing from scratch...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

...

Map:   0%|          | 0/2453 [00:00<?, ? examples/s]


Tokenization complete. Saving tokenized dataset to disk...


Saving the dataset (0/1 shards):   0%|          | 0/22072 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2453 [00:00<?, ? examples/s]

Dataset saved.

Final tokenized dataset structure:
DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 22072
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 2453
    })
})

Example of a single tokenized entry:
{'input_ids': [101, 177, 17112, 10957, 23746, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1], 'labels': [101, 10916, 12574, 43632, 22848, 23856, 10916, 12687, 58609, 102]}


In [ ]:
# --- 3. MODEL IMPLEMENTATION AND TRAINING (Definitive Corrected Version) ---
from transformers import TFAutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, create_optimizer
from datasets import load_dataset, DatasetDict, Dataset
import numpy as np
import tensorflow as tf # Make sure tensorflow is imported

# --- This part is to ensure the notebook can be run from top-to-bottom ---
# In a real run, these variables would already exist from the previous cells.
MODEL_CHECKPOINT = "Helsinki-NLP/opus-mt-en-ur"

# We must re-define the tokenizer and model objects in this cell
# if we intend to run it independently or after a restart.
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)
model = TFAutoModelForSeq2SeqLM.from_pretrained(MODEL_CHECKPOINT, from_pt=True)

# Assuming 'tokenized_datasets' exists from the previous cell. If not, you need to
# re-run the tokenization cell.

# --- Prepare the data for the model ---
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, return_tensors="tf")

tf_train_dataset = tokenized_datasets["train"].to_tf_dataset(
    columns=["input_ids", "attention_mask", "labels"],
    collate_fn=data_collator,
    shuffle=True,
    batch_size=16
)

tf_test_dataset = tokenized_datasets["test"].to_tf_dataset(
    columns=["input_ids", "attention_mask", "labels"],
    collate_fn=data_collator,
    shuffle=False,
    batch_size=16
)

# --- THE FIX: COMPILE THE MODEL USING THE HUGGING FACE OPTIMIZER ---

# 1. Calculate the number of training steps for the learning rate schedule.
num_train_epochs = 3
num_train_steps = len(tf_train_dataset) * num_train_epochs

# 2. Use the 'create_optimizer' helper from Hugging Face.
# This creates an AdamW optimizer with the correct learning rate decay (linear warmup and decay).
optimizer, schedule = create_optimizer(
    init_lr=2e-5,
    num_warmup_steps=0,
    num_train_steps=num_train_steps,
)

# 3. Compile the model with the created optimizer.
# The model will use its own internal loss function.
model.compile(optimizer=optimizer)


# --- Train the model ---
print("\nStarting model fine-tuning...")

history = model.fit(
    tf_train_dataset,
    validation_data=tf_test_dataset,
    epochs=num_train_epochs
)

print("Model fine-tuning finished.")

# --- CHECKPOINTING: Save the final model ---
SAVE_PATH = "./final_translation_model"
print(f"\nSaving the fine-tuned model to '{SAVE_PATH}'...")
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print("Model and tokenizer saved.")

All PyTorch model weights were used when initializing TFMarianMTModel.

All the weights of TFMarianMTModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFMarianMTModel for predictions without further training.



Starting model fine-tuning...
Epoch 1/3
1380/1380 [==============================] - 282s 149ms/step - loss: 1.4622 - val_loss: 1.1475
Epoch 2/3
1380/1380 [==============================] - 194s 140ms/step - loss: 1.0831 - val_loss: 1.0732
Epoch 3/3
1380/1380 [==============================] - 195s 141ms/step - loss: 0.9446 - val_loss: 1.0574
Model fine-tuning finished.

Saving the fine-tuned model to './final_translation_model'...


/usr/local/lib/python3.12/dist-packages/transformers/configuration_utils.py:461: UserWarning: Some non-default generation parameters are set in the model config. These should go into either a) `model.generation_config` (as opposed to `model.config`); OR b) a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model).This warning will become an exception in the future.
Non-default generation parameters: {'max_length': 512, 'num_beams': 4, 'bad_words_ids': [[62024]]}
  warnings.warn(


Model and tokenizer saved.


In [ ]:
# --- 4. EVALUATION AND ANALYSIS (Corrected) ---
# Install the 'evaluate' library, which now contains all metrics
!pip install -q evaluate

from transformers import pipeline, AutoTokenizer, TFAutoModelForSeq2SeqLM
import evaluate # <-- THE NEW IMPORT
from tqdm import tqdm
import numpy as np

# --- Part 1: Qualitative Evaluation (Example Translations) ---

# Load the fine-tuned model and tokenizer from our saved checkpoint
model_checkpoint = "./final_translation_model"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = TFAutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)

# Create a translation pipeline
translator = pipeline("translation", model=model, tokenizer=tokenizer)

print("--- Example Translations ---")
english_sentences = [
    "I am a student.",
    "The weather is nice today.",
    "What is your name?",
    "This is a difficult assignment but I am learning a lot."
]

for sentence in english_sentences:
    urdu_translation = translator(sentence)
    print(f"English: {sentence}")
    print(f"Urdu (Model): {urdu_translation[0]['translation_text']}")
    print("-" * 30)

# --- Part 2: Quantitative Evaluation (BLEU Score) ---

# THE FIX IS HERE: Load the BLEU metric from the 'evaluate' library
bleu_metric = evaluate.load("sacrebleu")

# We need to generate predictions for the entire test set
all_preds = []
all_labels = []

print("\n--- Calculating BLEU score on the test set ---")
print("(This may take a few minutes...)")

# We need to get the raw text from the original dataset split, not the tokenized one
test_texts = dataset["test"] # 'dataset' is from our preprocessing cell

# Iterate through the test set to generate predictions
for example in tqdm(test_texts):
    # Get the English input and the true Urdu label
    input_text = example["English"]
    label_text = example["Urdu"]

    # Generate translation
    # Adding max_length to prevent overly long, repetitive translations
    prediction = translator(input_text, max_length=128)[0]['translation_text']

    # Sacrebleu expects predictions as a list of strings, and labels as a list of lists of strings
    all_preds.append(prediction)
    all_labels.append([label_text])

# Compute the final BLEU score
final_bleu = bleu_metric.compute(predictions=all_preds, references=all_labels)

print("\n--- Final Quantitative Results ---")
print(f"BLEU Score on Test Set: {final_bleu['score']:.2f}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.9 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")
All model checkpoint layers were used when initializing TFMarianMTModel.

All the layers of TFMarianMTModel were initialized from the model checkpoint at ./final_translation_model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFMarianMTModel for predictions without further training.
TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.
Device set to use 0
TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.


--- Example Translations ---
English: I am a student.
Urdu (Model): میں طالب علم ہوں۔
------------------------------
English: The weather is nice today.
Urdu (Model): موسم اچھا ہے
------------------------------
English: What is your name?
Urdu (Model): تمہارا نام کیا ہے؟
------------------------------
English: This is a difficult assignment but I am learning a lot.
Urdu (Model): یہ ایک مشکل کام ہے لیکن میں بہت کچھ سیکھ رہا ہوں۔
------------------------------



--- Calculating BLEU score on the test set ---
(This may take a few minutes...)


100%|██████████| 2453/2453 [1:35:33<00:00,  2.34s/it]


--- Final Quantitative Results ---
BLEU Score on Test Set: 49.54
